In [1]:
# ============================================================
# MVC Project - Task 7: MLP Implementation on MNIST
# Student: Ahmad Mustafa | Roll No: 25I-2565 | Section: DS-A
# ============================================================

# Cell 1 - Import Libraries
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)

# ============================================================
# Cell 2 - Load MNIST Dataset
# ============================================================
import urllib.request
import os

# Download MNIST if not present
mnist_path = 'mnist.npz'
if not os.path.exists(mnist_path):
    print("Downloading MNIST dataset...")
    url = 'https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz'
    urllib.request.urlretrieve(url, mnist_path)
    print("Download complete!")

data = np.load(mnist_path)
X_train = data['x_train'].reshape(-1, 784) / 255.0  # (60000, 784)
Y_train = data['y_train']                            # (60000,)
X_test  = data['x_test'].reshape(-1, 784) / 255.0   # (10000, 784)
Y_test  = data['y_test']                             # (10000,)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"Classes: {np.unique(Y_train)}")

# One-hot encode labels
def one_hot(y, num_classes=10):
    m = len(y)
    oh = np.zeros((m, num_classes))
    oh[np.arange(m), y] = 1
    return oh

Y_train_oh = one_hot(Y_train)
Y_test_oh  = one_hot(Y_test)

# ============================================================
# Cell 3 - Network Architecture
# Layer 0: Input  - 784 neurons
# Layer 1: Hidden - 128 neurons (Sigmoid)
# Layer 2: Hidden -  64 neurons (Sigmoid)
# Layer 3: Output -  10 neurons (Sigmoid)
# ============================================================

# Initialize weights randomly in [-0.5, 0.5] and biases to 0
W1 = np.random.uniform(-0.5, 0.5, (784, 128))
b1 = np.zeros((1, 128))
W2 = np.random.uniform(-0.5, 0.5, (128, 64))
b2 = np.zeros((1, 64))
W3 = np.random.uniform(-0.5, 0.5, (64, 10))
b3 = np.zeros((1, 10))

print(f"W1: {W1.shape}, W2: {W2.shape}, W3: {W3.shape}")

# ============================================================
# Cell 4 - Step 1: Sigmoid Activation (from Task 1)
# ============================================================
def sigmoid(z):
    # Clip to avoid overflow
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(a):
    # Derived in Task 3: sigma'(z) = a(1-a)
    return a * (1 - a)

# ============================================================
# Cell 5 - Step 2: Forward Pass (from Task 1)
# ============================================================
def forward_pass(X, W1, b1, W2, b2, W3, b3):
    # Layer 1
    Z1 = X @ W1 + b1          # weighted sum
    A1 = sigmoid(Z1)           # sigmoid activation

    # Layer 2
    Z2 = A1 @ W2 + b2
    A2 = sigmoid(Z2)

    # Output layer
    Z3 = A2 @ W3 + b3
    A3 = sigmoid(Z3)

    return Z1, A1, Z2, A2, Z3, A3

# ============================================================
# Cell 6 - Step 3: MSE Loss (from Task 2)
# ============================================================
def mse_loss(Y_true, Y_pred):
    # Mean over all samples and all output neurons
    return np.mean((Y_true - Y_pred) ** 2)

# ============================================================
# Cell 7 - Step 4: Backpropagation (from Task 3)
# ============================================================
def backpropagation(X, Y_true, Z1, A1, Z2, A2, Z3, A3, W1, W2, W3):
    m = X.shape[0]

    # Output layer delta
    delta3 = -2 * (Y_true - A3) * sigmoid_derivative(A3)

    # Gradients for W3, b3
    dW3 = (A2.T @ delta3) / m
    db3 = np.mean(delta3, axis=0, keepdims=True)

    # Hidden layer 2 delta
    delta2 = (delta3 @ W3.T) * sigmoid_derivative(A2)

    # Gradients for W2, b2
    dW2 = (A1.T @ delta2) / m
    db2 = np.mean(delta2, axis=0, keepdims=True)

    # Hidden layer 1 delta
    delta1 = (delta2 @ W2.T) * sigmoid_derivative(A1)

    # Gradients for W1, b1
    dW1 = (X.T @ delta1) / m
    db1 = np.mean(delta1, axis=0, keepdims=True)

    return dW1, db1, dW2, db2, dW3, db3

# ============================================================
# Cell 8 - Step 5: Weight Update (from Task 4)
# ============================================================
def update_weights(W1, b1, W2, b2, W3, b3,
                   dW1, db1, dW2, db2, dW3, db3,
                   learning_rate):
    W1 = W1 - learning_rate * dW1
    b1 = b1 - learning_rate * db1
    W2 = W2 - learning_rate * dW2
    b2 = b2 - learning_rate * db2
    W3 = W3 - learning_rate * dW3
    b3 = b3 - learning_rate * db3
    return W1, b1, W2, b2, W3, b3

# ============================================================
# Cell 9 - Training Loop (Mini-Batch GD, from Task 5)
# ============================================================
learning_rate = 0.1
epochs = 20
batch_size = 32
loss_history = []

print("Starting training...\n")

for epoch in range(epochs):
    # Shuffle training data
    idx = np.random.permutation(X_train.shape[0])
    X_shuf = X_train[idx]
    Y_shuf = Y_train_oh[idx]

    # Mini-batch loop
    for start in range(0, X_train.shape[0], batch_size):
        X_batch = X_shuf[start : start + batch_size]
        Y_batch = Y_shuf[start : start + batch_size]

        # Forward pass
        Z1, A1, Z2, A2, Z3, A3 = forward_pass(X_batch, W1, b1, W2, b2, W3, b3)

        # Backpropagation
        dW1, db1, dW2, db2, dW3, db3 = backpropagation(
            X_batch, Y_batch, Z1, A1, Z2, A2, Z3, A3, W1, W2, W3)

        # Update weights
        W1, b1, W2, b2, W3, b3 = update_weights(
            W1, b1, W2, b2, W3, b3,
            dW1, db1, dW2, db2, dW3, db3,
            learning_rate)

    # Record epoch loss on full training set
    _, _, _, _, _, A3_full = forward_pass(X_train, W1, b1, W2, b2, W3, b3)
    epoch_loss = mse_loss(Y_train_oh, A3_full)
    loss_history.append(epoch_loss)
    print(f"Epoch {epoch+1:2d}/{epochs} | Loss: {epoch_loss:.4f}")

# ============================================================
# Cell 10 - Output 1: Loss Curve
# ============================================================
plt.figure(figsize=(9, 5))
plt.plot(range(1, epochs+1), loss_history, 'b-o', linewidth=2, markersize=5)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('MSE Loss', fontsize=12)
plt.title('Training Loss Curve - MLP on MNIST (25I-2565)', fontsize=13)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=150)
plt.show()
print("Loss curve saved!")

# ============================================================
# Cell 11 - Output 2: Test Accuracy
# ============================================================
_, _, _, _, _, A3_test = forward_pass(X_test, W1, b1, W2, b2, W3, b3)
Y_pred = np.argmax(A3_test, axis=1)
accuracy = np.mean(Y_pred == Y_test) * 100
print(f"\nFinal Test Accuracy: {accuracy:.2f}%")

# ============================================================
# Cell 12 - Output 3: Sample Predictions (one per digit class)
# ============================================================
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes = axes.flatten()

for digit in range(10):
    # Find first test image of this digit
    idx = np.where(Y_test == digit)[0][0]
    img = X_test[idx].reshape(28, 28)
    true_label = Y_test[idx]
    pred_label = Y_pred[idx]

    axes[digit].imshow(img, cmap='gray')
    color = 'green' if true_label == pred_label else 'red'
    axes[digit].set_title(f"True: {true_label} | Pred: {pred_label}", color=color, fontsize=10)
    axes[digit].axis('off')

plt.suptitle('Sample Predictions - One Per Digit Class (25I-2565)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150)
plt.show()
print("Sample predictions saved!")

X_train shape: (60000, 784)
X_test shape:  (10000, 784)
Classes: [0 1 2 3 4 5 6 7 8 9]
W1: (784, 128), W2: (128, 64), W3: (64, 10)
Starting training...

Epoch  1/20 | Loss: 0.0296
Epoch  2/20 | Loss: 0.0208
Epoch  3/20 | Loss: 0.0172
Epoch  4/20 | Loss: 0.0153
Epoch  5/20 | Loss: 0.0138
Epoch  6/20 | Loss: 0.0127
Epoch  7/20 | Loss: 0.0119
Epoch  8/20 | Loss: 0.0111
Epoch  9/20 | Loss: 0.0105
Epoch 10/20 | Loss: 0.0100
Epoch 11/20 | Loss: 0.0095
Epoch 12/20 | Loss: 0.0091
Epoch 13/20 | Loss: 0.0088
Epoch 14/20 | Loss: 0.0084
Epoch 15/20 | Loss: 0.0081
Epoch 16/20 | Loss: 0.0078
Epoch 17/20 | Loss: 0.0076
Epoch 18/20 | Loss: 0.0073
Epoch 19/20 | Loss: 0.0071
Epoch 20/20 | Loss: 0.0069


Loss curve saved!

Final Test Accuracy: 95.54%


Sample predictions saved!
